In [8]:
import pandas as pd
import re

def checkpoint_sort_key(s):
    # epoch_1_batch_0.pt -> (1, 0), epoch_1.pt -> (1, 10000), epoch_2.pt -> (2, 10000)
    m = re.match(r'epoch_(\d+)(?:_batch_(\d+))?', s)
    if m:
        epoch = int(m.group(1))
        if m.group(2):
            batch = int(m.group(2))
        else:
            batch = 10000  # ensure epoch_N.pt is after all epoch_N_batch_M.pt
        return (epoch, batch)
    return (9999, 9999)

df = pd.read_csv('results_test_orc_train_RNNModel_base_data.csv')
df['sort_key'] = df['checkpoint'].map(checkpoint_sort_key)
df = df.sort_values('sort_key').reset_index(drop=True)
print(df[['checkpoint', 'perplexity']])

              checkpoint    perplexity
0     epoch_1_batch_0.pt  46730.328125
1   epoch_1_batch_100.pt   1010.864807
2   epoch_1_batch_200.pt    727.782349
3   epoch_1_batch_300.pt    646.132141
4   epoch_1_batch_400.pt    619.372925
5   epoch_1_batch_500.pt    621.973572
6   epoch_1_batch_600.pt    646.007996
7   epoch_1_batch_700.pt    588.724731
8   epoch_1_batch_800.pt    578.864136
9   epoch_1_batch_900.pt    561.059937
10            epoch_1.pt    603.693481
11            epoch_2.pt    580.219604
12            epoch_3.pt    550.498596
13            epoch_4.pt    578.065308
14            epoch_5.pt    634.695007
15            epoch_6.pt    705.335388
16            epoch_7.pt    764.151306
17            epoch_8.pt    866.402405
18            epoch_9.pt    920.082947
19           epoch_10.pt    976.369446


In [4]:
import glob
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import re

colors = {
    'base_data': 'rgb(173,216,230)',
    'freq_4': 'rgb(135,206,250)',
    'freq_6': 'rgb(70,130,180)',
    'freq_8': 'rgb(100,149,237)',
    'freq_16': 'rgb(65,105,225)',
    'freq_32': 'rgb(0,0,139)'
}
linestyles = {'test': 'solid', 'test_orc': 'dot'}
files = sorted(glob.glob('results_test_*.csv'))

def checkpoint_sort_key(s):
    m = re.match(r'epoch_(\d+)(?:_batch_(\d+))?', s)
    if m:
        epoch = int(m.group(1))
        batch = int(m.group(2)) if m.group(2) else 10000
        return (epoch, batch)
    return (9999, 9999)

fig = go.Figure()
for f in files:
    df = pd.read_csv(f)
    df = df[df['checkpoint'] != 'epoch_1_batch_0.pt'] 
    df['sort_key'] = df['checkpoint'].map(checkpoint_sort_key)
    df = df.sort_values('sort_key').reset_index(drop=True)
    x = df['checkpoint'].str.replace('.pt', '', regex=False).values
    y = np.log(df['perplexity'].values)
    for key in colors:
        if key in f:
            color = colors[key]
    style = 'test_orc' if 'test_orc' in f else 'test'
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers',
        name=f.split('results_')[1].replace('.csv',''),
        line=dict(color=color, dash=linestyles[style])
    ))
fig.update_layout(
    title='Cross-Entropy Loss vs. Checkpoint',
    xaxis_title='Checkpoint',
    yaxis_title='Cross-Entropy Loss (log perplexity)',
    legend_title='Model/Test',
    width=1200,
    height=600
)
fig.show()

In [3]:
import glob
import pandas as pd
import plotly.graph_objects as go
import re

colors = {
    'base_data': 'rgb(173,216,230)',
    'freq_4': 'rgb(135,206,250)',
    'freq_6': 'rgb(70,130,180)',
    'freq_8': 'rgb(100,149,237)',
    'freq_16': 'rgb(65,105,225)',
    'freq_32': 'rgb(0,0,139)'
}
files = sorted(glob.glob('results_test_*.csv'))
files = [f for f in files if 'test_orc' not in f]  # Only test, not test_orc

def checkpoint_sort_key(s):
    m = re.match(r'epoch_(\d+)(?:_batch_(\d+))?', s)
    if m:
        epoch = int(m.group(1))
        batch = int(m.group(2)) if m.group(2) else 10000
        return (epoch, batch)
    return (9999, 9999)

fig = go.Figure()
for f in files:
    df = pd.read_csv(f)
    df = df[df['checkpoint'] != 'epoch_1_batch_0.pt']  # Exclude epoch_1_batch_0
    df['sort_key'] = df['checkpoint'].map(checkpoint_sort_key)
    df = df.sort_values('sort_key').reset_index(drop=True)
    x = df['checkpoint'].str.replace('.pt', '', regex=False).values
    y = df['perplexity'].values
    for key in colors:
        if key in f:
            color = colors[key]
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers',
        name=f.split('results_')[1].replace('.csv',''),
        line=dict(color=color)
    ))
fig.update_layout(
    title='Perplexity vs. Checkpoint (Test Dataset)',
    xaxis_title='Checkpoint',
    yaxis_title='Perplexity',
    legend_title='Model/Test',
    width=1200,
    height=600
)
fig.show()